# Day 8: Data Preprocessing and Data Cleaning

This notebook cleans the Day 7 Weather History dataset with Pandas and NumPy. It keeps only necessary transformations and records the results before saving a new CSV file.

## Import libraries and load the dataset

The dataset is loaded with a relative path so this notebook can be run from the `Day-8` folder.

In [1]:
import numpy as np
import pandas as pd

data_path = "../Day-7/weatherHistory.csv"
weather_df = pd.read_csv(data_path)
weather_df.head()

,Formatted Date,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars),Daily Summary
0,2006-04-01 00:00:00.000 +0200,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,0.0,1015.13,Partly cloudy throughout the day.
1,2006-04-01 01:00:00.000 +0200,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,0.0,1015.63,Partly cloudy throughout the day.
2,2006-04-01 02:00:00.000 +0200,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,0.0,1015.94,Partly cloudy throughout the day.
3,2006-04-01 03:00:00.000 +0200,Partly Cloudy,rain,8.288889,5.944444,0.83,14.1036,269.0,15.8263,0.0,1016.41,Partly cloudy throughout the day.
4,2006-04-01 04:00:00.000 +0200,Mostly Cloudy,rain,8.755556,6.977778,0.83,11.0446,259.0,15.8263,0.0,1016.51,Partly cloudy throughout the day.


## Inspect the original dataset

Review the size, columns, and data types before making changes.

In [2]:
print("Dataset shape:", weather_df.shape)
print("\nColumn names:")
print(weather_df.columns.tolist())
print("\nData types:")
print(weather_df.dtypes)

Dataset shape: (96453, 12)

Column names:
['Formatted Date', 'Summary', 'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)', 'Loud Cover', 'Pressure (millibars)', 'Daily Summary']

Data types:
Formatted Date               object
Summary                      object
Precip Type                  object
Temperature (C)             float64
Apparent Temperature (C)    float64
Humidity                    float64
Wind Speed (km/h)           float64
Wind Bearing (degrees)      float64
Visibility (km)             float64
Loud Cover                  float64
Pressure (millibars)        float64
Daily Summary                object
dtype: object


## Check missing values

Both counts and percentages are shown. This helps decide whether a value should be filled or left unchanged.

In [3]:
missing_before = pd.DataFrame({
    "missing_count": weather_df.isna().sum(),
    "missing_percentage": (weather_df.isna().mean() * 100).round(2)
})
missing_before[missing_before["missing_count"] > 0]

,missing_count,missing_percentage
Precip Type,517,0.54


## Record the starting values

These values are used for the before-and-after comparison. Exact duplicate rows are counted before they are removed.

In [4]:
before_rows = len(weather_df)
before_duplicates = weather_df.duplicated().sum()
before_missing_total = weather_df.isna().sum().sum()
before_dtypes = weather_df.dtypes.copy()

print(f"Rows before cleaning: {before_rows}")
print(f"Exact duplicate rows: {before_duplicates}")
print(f"Total missing values: {before_missing_total}")

Rows before cleaning: 96453
Exact duplicate rows: 24
Total missing values: 517


## Remove exact duplicates and standardize column names

Only records that match in every column are removed. Column names are changed to lowercase snake_case to make them easier to use in Python.

In [5]:
weather_df = weather_df.drop_duplicates().copy()

weather_df = weather_df.rename(columns={
    "Formatted Date": "formatted_date",
    "Summary": "summary",
    "Precip Type": "precip_type",
    "Temperature (C)": "temperature_c",
    "Apparent Temperature (C)": "apparent_temperature_c",
    "Humidity": "humidity",
    "Wind Speed (km/h)": "wind_speed_km_h",
    "Wind Bearing (degrees)": "wind_bearing_degrees",
    "Visibility (km)": "visibility_km",
    "Loud Cover": "loud_cover",
    "Pressure (millibars)": "pressure_millibars",
    "Daily Summary": "daily_summary"
})

before_dtypes.index = weather_df.columns
print(weather_df.columns.tolist())

['formatted_date', 'summary', 'precip_type', 'temperature_c', 'apparent_temperature_c', 'humidity', 'wind_speed_km_h', 'wind_bearing_degrees', 'visibility_km', 'loud_cover', 'pressure_millibars', 'daily_summary']


## Convert date and numeric columns

Invalid dates or numeric values would become missing values rather than causing the notebook to stop. The date conversion uses UTC so offsets in the source data are handled consistently.

In [6]:
weather_df["formatted_date"] = pd.to_datetime(
    weather_df["formatted_date"], errors="coerce", utc=True
)

numeric_columns = weather_df.select_dtypes(include=[np.number]).columns.tolist()

for column in numeric_columns:
    weather_df[column] = pd.to_numeric(weather_df[column], errors="coerce")

print(f"Invalid or missing dates after conversion: {weather_df['formatted_date'].isna().sum()}")

Invalid or missing dates after conversion: 0


## Handle missing values carefully

`precip_type` is the only source column with missing values, so it is filled with its most common value (mode). Numerical columns are checked individually and would use their median only if missing values existed. The text columns `summary` and `daily_summary` are left unchanged because they have no missing values. The date column is also left unchanged after conversion so invalid dates, if any, remain visible rather than being invented.

In [7]:
precip_type_mode = weather_df["precip_type"].mode()
if weather_df["precip_type"].isna().any() and not precip_type_mode.empty:
    weather_df["precip_type"] = weather_df["precip_type"].fillna(precip_type_mode.iloc[0])

for column in numeric_columns:
    if weather_df[column].isna().any():
        weather_df[column] = weather_df[column].fillna(weather_df[column].median())

missing_after = pd.DataFrame({
    "missing_count": weather_df.isna().sum(),
    "missing_percentage": (weather_df.isna().mean() * 100).round(2)
})
missing_after[missing_after["missing_count"] > 0]

,missing_count,missing_percentage


## Compare results before and after cleaning

The comparison confirms that row loss is limited to exact duplicates and shows the final data types.

In [8]:
comparison = pd.DataFrame({
    "before": [before_rows, before_duplicates, before_missing_total],
    "after": [len(weather_df), weather_df.duplicated().sum(), weather_df.isna().sum().sum()]
}, index=["rows", "exact_duplicates", "total_missing_values"])

print(comparison)

dtype_comparison = pd.DataFrame({
    "before": before_dtypes.astype(str),
    "after": weather_df.dtypes.astype(str)
})
print("\nData types before and after:")
print(dtype_comparison)

                      before  after
rows                   96453  96429
exact_duplicates          24      0
total_missing_values     517      0

Data types before and after:
                         before                after
formatted_date           object  datetime64[ns, UTC]
summary                  object               object
precip_type              object               object
temperature_c           float64              float64
apparent_temperature_c  float64              float64
humidity                float64              float64
wind_speed_km_h         float64              float64
wind_bearing_degrees    float64              float64
visibility_km           float64              float64
loud_cover              float64              float64
pressure_millibars      float64              float64
daily_summary            object               object


## Save the cleaned dataset

The cleaned CSV is saved in `Day-8` without the Pandas index.

In [9]:
output_path = "cleaned_weather.csv"
weather_df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to {output_path} with {len(weather_df)} rows.")

Saved cleaned dataset to cleaned_weather.csv with 96429 rows.
